In [10]:
!pip install pydicom opencv-python-headless scikit-learn --quiet
print("Dependencies installed.")

Dependencies installed.


In [11]:
import os
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import roc_auc_score
from copy import deepcopy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"All imports successful.")
print(f"Device: {device}")

All imports successful.
Device: cuda


In [12]:
def get_model():
    model = models.densenet121(weights=None)
    model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.classifier = nn.Linear(model.classifier.in_features, 2)
    return model

def get_parameters(model):
    return [val.cpu().numpy() for _, val in model.state_dict().items()]

def set_parameters(model, parameters):
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict  = {k: torch.tensor(v) for k, v in params_dict}
    model.load_state_dict(state_dict, strict=True)

def fedavg(global_params, client_params_list, client_sizes):
    """
    Federated Averaging — combines weight updates from all hospital nodes.
    Each hospital's contribution is weighted by its dataset size.
    No raw data is shared — only the learned weight updates.
    """
    total = sum(client_sizes)
    averaged = []
    for layer_idx in range(len(global_params)):
        layer_avg = sum(
            (client_params_list[i][layer_idx] * client_sizes[i] / total)
            for i in range(len(client_params_list))
        )
        averaged.append(layer_avg)
    return averaged

print("Model and FedAvg functions defined.")
print(f"Parameters in model: {sum(p.numel() for p in get_model().parameters()):,}")

Model and FedAvg functions defined.
Parameters in model: 6,949,634


In [13]:
class SyntheticXrayDataset(Dataset):
    """
    Synthetic dataset simulating preprocessed chest X-ray tensors.
    Each hospital node gets a different data partition with slight
    distribution shifts to simulate real-world institutional variation.
    """
    def __init__(self, n_samples=500, hospital_id=0, seed=42):
        random.seed(seed + hospital_id)
        np.random.seed(seed + hospital_id)

        pneumonia_rate   = 0.2 + (hospital_id * 0.05)
        self.labels      = np.random.binomial(1, pneumonia_rate, n_samples)

        self.images = []
        for label in self.labels:
            img = np.random.normal(0.4 + (0.1 * label), 0.15, (1, 224, 224))
            img = np.clip(img, 0, 1).astype(np.float32)
            self.images.append(img)

        self.images = np.array(self.images)
        print(f"   Hospital {hospital_id + 1}: {n_samples} images, "
              f"{self.labels.sum()} pneumonia ({pneumonia_rate*100:.0f}%)")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (torch.tensor(self.images[idx], dtype=torch.float32),
                torch.tensor(self.labels[idx], dtype=torch.long))


NUM_HOSPITALS = 5
SAMPLES_PER_HOSPITAL = 500
BATCH_SIZE = 32

print("Creating simulated hospital datasets...")
hospital_datasets = [
    SyntheticXrayDataset(n_samples=SAMPLES_PER_HOSPITAL, hospital_id=i)
    for i in range(NUM_HOSPITALS)
]
print(f"\n{NUM_HOSPITALS} hospital nodes created.")
print(f"Total simulated images: {NUM_HOSPITALS * SAMPLES_PER_HOSPITAL:,}")

Creating simulated hospital datasets...
   Hospital 1: 500 images, 106 pneumonia (20%)
   Hospital 2: 500 images, 131 pneumonia (25%)
   Hospital 3: 500 images, 154 pneumonia (30%)
   Hospital 4: 500 images, 172 pneumonia (35%)
   Hospital 5: 500 images, 185 pneumonia (40%)

5 hospital nodes created.
Total simulated images: 2,500


In [14]:
def train_hospital(model, dataset, epochs=1):
    """
    Train a hospital's local model on its private data.
    Only the resulting weight updates leave the hospital — never the data.
    """
    n       = len(dataset)
    n_train = int(0.8 * n)
    indices = list(range(n))
    random.shuffle(indices)

    train_loader = DataLoader(
        Subset(dataset, indices[:n_train]),
        batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        Subset(dataset, indices[n_train:]),
        batch_size=BATCH_SIZE, shuffle=False
    )

    n_pos    = int(dataset.labels.sum())
    n_neg    = n - n_pos
    weight   = torch.tensor([1.0, n_neg / max(n_pos, 1)]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()
    for epoch in range(epochs):
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), lbls)
            loss.backward()
            optimizer.step()

    # Evaluate on local validation set
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            out = model(imgs)
            all_probs.extend(F.softmax(out, dim=1)[:,1].cpu().numpy())
            all_labels.extend(lbls.cpu().numpy())

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0

    return get_parameters(model), n_train, auc


print("Hospital training function defined.")
print("Each hospital trains locally and returns only weight updates.")

Hospital training function defined.
Each hospital trains locally and returns only weight updates.


In [15]:
NUM_ROUNDS = 5

print("Starting federated learning simulation...")
print(f"   Hospitals : {NUM_HOSPITALS}")
print(f"   Rounds    : {NUM_ROUNDS}")
print(f"   Strategy  : FedAvg (custom implementation)")
print("-" * 55)

# Initialise global model
global_model = get_model().to(device)
global_params = get_parameters(global_model)

round_metrics = []

for fed_round in range(1, NUM_ROUNDS + 1):
    start = time.time()
    print(f"\nRound {fed_round}/{NUM_ROUNDS}")

    client_params_list = []
    client_sizes       = []
    client_aucs        = []

    # Each hospital trains on its local data
    for hospital_id in range(NUM_HOSPITALS):
        local_model = get_model().to(device)
        set_parameters(local_model, global_params)

        local_params, n_samples, local_auc = train_hospital(
            local_model, hospital_datasets[hospital_id], epochs=1
        )

        client_params_list.append(local_params)
        client_sizes.append(n_samples)
        client_aucs.append(local_auc)
        print(f"   Hospital {hospital_id + 1} | "
              f"Samples: {n_samples} | "
              f"Local AUC: {local_auc:.4f}")

    # Coordinator aggregates weight updates via FedAvg
    global_params = fedavg(global_params, client_params_list, client_sizes)
    set_parameters(global_model, global_params)

    avg_auc = np.mean(client_aucs)
    elapsed = time.time() - start
    round_metrics.append({"round": fed_round, "avg_auc": avg_auc})

    print(f"   Global model updated via FedAvg")
    print(f"   Average AUC across hospitals: {avg_auc:.4f}")
    print(f"   Round time: {elapsed:.1f}s")
    print("-" * 55)

print("\nFederated learning simulation complete.")
print(f"   Rounds completed : {NUM_ROUNDS}")
print(f"   Hospitals        : {NUM_HOSPITALS}")
print(f"   Final avg AUC    : {round_metrics[-1]['avg_auc']:.4f}")

Starting federated learning simulation...
   Hospitals : 5
   Rounds    : 5
   Strategy  : FedAvg (custom implementation)
-------------------------------------------------------

Round 1/5
   Hospital 1 | Samples: 400 | Local AUC: 0.0000
   Hospital 2 | Samples: 400 | Local AUC: 1.0000
   Hospital 3 | Samples: 400 | Local AUC: 1.0000
   Hospital 4 | Samples: 400 | Local AUC: 1.0000
   Hospital 5 | Samples: 400 | Local AUC: 1.0000
   Global model updated via FedAvg
   Average AUC across hospitals: 0.8000
   Round time: 24.1s
-------------------------------------------------------

Round 2/5
   Hospital 1 | Samples: 400 | Local AUC: 1.0000
   Hospital 2 | Samples: 400 | Local AUC: 1.0000
   Hospital 3 | Samples: 400 | Local AUC: 1.0000
   Hospital 4 | Samples: 400 | Local AUC: 1.0000
   Hospital 5 | Samples: 400 | Local AUC: 1.0000
   Global model updated via FedAvg
   Average AUC across hospitals: 1.0000
   Round time: 21.9s
-------------------------------------------------------

Round

In [16]:
# Save the globally trained federated model
FEDERATED_MODEL_PATH = 'federated_global_model.pth'
torch.save(global_model.state_dict(), FEDERATED_MODEL_PATH)

print("Federated training summary")
print("=" * 55)
print(f"{'Round':<10} {'Avg AUC':<15}")
print("-" * 25)
for m in round_metrics:
    print(f"{m['round']:<10} {m['avg_auc']:.4f}")
print("-" * 25)
print(f"\nFinal global model AUC : {round_metrics[-1]['avg_auc']:.4f}")
print(f"Hospitals participated : {NUM_HOSPITALS}")
print(f"Total rounds           : {NUM_ROUNDS}")
print(f"Total images trained on: {NUM_HOSPITALS * SAMPLES_PER_HOSPITAL:,}")
print(f"Patient data shared    : 0 bytes")
print(f"Model saved to         : {FEDERATED_MODEL_PATH}")
print("\nPhase 3 complete. Ready for Phase 4 - Edge Deployment.")

Federated training summary
Round      Avg AUC        
-------------------------
1          0.8000
2          1.0000
3          1.0000
4          1.0000
5          1.0000
-------------------------

Final global model AUC : 1.0000
Hospitals participated : 5
Total rounds           : 5
Total images trained on: 2,500
Patient data shared    : 0 bytes
Model saved to         : federated_global_model.pth

Phase 3 complete. Ready for Phase 4 - Edge Deployment.


In [17]:
from google.colab import files
files.download('federated_global_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>